# Data Science Project 1: Data Wrangling Practical
**Duration:** 1 Hour, 15 Minutes  
**Format:** Open-notes code implementation (Python/Pandas)

### **Project Scenario**
You have been given a raw slice of **Glassdoor Data Science Job Postings**. The dataset contains a variety of small errors, missing variables, and split locations. Your goal is to clean this main dataset, create engineered features, and merge it with a regional mapping file so the analytics team can use it.

## Load the data
Download the data from the Blackboard to your computer frist, then load your libraries and the datasets required for this exam here.

In [41]:
# import all libaries and load your datasets your here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('glassdoor_jobs.csv')
df2 = pd.read_csv('state_regions.csv')

sns.set(style='whitegrid')

In [42]:
df.head()

,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.1,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.2,ManTech\n4.2,"Chantilly, VA",1968,Research & Development
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,-1.0,Analysis Group\n3.8,"Boston, MA",1981,Consulting
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.5,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing


In [43]:
df2.head()

,State_Abbreviation,US_Region
0,IL,Midwest
1,TX,South
2,CA,West
3,NY,Northeast
4,MA,Northeast


## Part 1: Missing Data & Filtering (30 Points)

### **Task 1.1 (Standardizing Nulls)**
The `Rating` column represents missing values as `-1`. Replace all occurrences of `-1` with the dataset's overall **average (mean)** rating value.

In [44]:
# Write your solution for Task 1.1 here
df.loc[df['Rating'] == -1, 'Rating'] = df['Rating'].mean()

### **Task 1.2 (Handling Blanks)**
The `Industry` column has several completely blank or empty cells. Fill these missing values with the string text `"Unknown"`.

In [45]:
# Write your solution for Task 1.2 here
df['Industry'] =df['Industry'].fillna('Unknown')
df['Industry'].head(40)

0                           Insurance Carriers
1                       Research & Development
2                                   Consulting
3        Electrical & Electronic Manufacturing
4                      Advertising & Marketing
5                                      Unknown
6                    Biotech & Pharmaceuticals
7     Consumer Electronics & Appliances Stores
8                 Computer Hardware & Software
9                                      Unknown
10     Enterprise Software & Network Solutions
11                                 IT Services
12                                      Energy
13                      Chemical Manufacturing
14                                  Consulting
15                            Federal Agencies
16                                    Internet
17                      Chemical Manufacturing
18     Enterprise Software & Network Solutions
19            Health Care Services & Hospitals
20                Computer Hardware & Software
21           

### **Task 1.3 (Data Filtering)**
A few rows contain placeholder or invalid values in the `Founded` year column (values less than or equal to `0`). **Filter the dataset** to permanently remove these invalid rows.

In [51]:
# Write your solution for Task 1.3 here
df= df[df['Founded']>0]
df.head()

,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry,Job_City,Job_State,Industry_Avg_Rating,Rating_Above_Industry_Avg
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.100000,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers,New York,NY,3.549599,-0.449599
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.200000,ManTech\n4.2,"Chantilly, VA",1968,Research & Development,Chantilly,VA,3.791778,0.408222
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,2.984226,Analysis Group\n3.8,"Boston, MA",1981,Consulting,Boston,MA,3.635794,-0.651567
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.500000,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing,Newton,MA,3.242113,0.257887
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.900000,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing,New York,NY,3.539615,-0.639615


## Part 2: String Manipulations (20 Points)

### **Task 2.1 (Location Splitting)**
The `Location` column lists locations as `"City, State"` (e.g., `"Chicago, IL"` or `"Austin, TX"`). Split this column into two separate, brand-new columns: `Job_City` and `Job_State`.

In [47]:
# Write your solution for Task 2.1 here
df['Job_City'] = df['Location'].str.extract(r"^(.*?)(?=:|,|$)")
df['Job_State'] = df['Location'].str[-2:]

### **Task 2.2 (Whitespace Cleaning)**
The new `Job_State` column might contain accidental leading or trailing empty spaces from the split. Apply a string cleaning function (like `.str.strip()`) to ensure no accidental spaces remain.

In [48]:
# Write your solution for Task 2.2 here
df['Job_State'] = df['Job_State'].str.strip('')

## Part 3: Feature Engineering with Grouped Transformations (20 Points)

### **Task 3.1 (Grouped Transform)**
The recruiting team wants to identify jobs that offer a higher-than-average user rating relative to their specific field. 

Use `.groupby()` and `.transform()` to calculate the **average (mean) rating for each `Industry`**, broadcasting it back as a new column named `Industry_Avg_Rating`. Then, create a column called `Rating_Above_Industry_Avg` by subtracting `Industry_Avg_Rating` from the job's original `Rating`.

In [49]:
# Write your solution for Task 3.1 here
df['Industry_Avg_Rating'] = df.groupby('Industry')['Rating'].transform('mean')
df['Rating_Above_Industry_Avg'] = df['Rating'] - df['Industry_Avg_Rating']

## Part 4: Merging & Aggregating Data (30 Points)

### **Task 4.1 (Dataset Merging)**
You are provided a second, clean reference CSV file named `state_regions.csv` which contains two columns: `State_Abbreviation` and `US_Region` (e.g., *IL, Midwest*). 

Perform a **left merge** to add the `US_Region` column to your main job postings dataframe by matching your new `Job_State` column with the reference file's `State_Abbreviation` column.

In [56]:
# Write your solution for Task 4.1 here
df = df.merge(df2[['State_Abbreviation','US_Region']],left_on='Job_State',right_on='State_Abbreviation',how='left')
df.head()

,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry,Job_City,Job_State,Industry_Avg_Rating,Rating_Above_Industry_Avg,State_Abbreviation_x,US_Region_x,State_Abbreviation_y,US_Region_y,State_Abbreviation,US_Region
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.100000,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers,New York,NY,3.549599,-0.449599,NY,Northeast,NY,Northeast,NY,Northeast
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.200000,ManTech\n4.2,"Chantilly, VA",1968,Research & Development,Chantilly,VA,3.791778,0.408222,VA,South,VA,South,VA,South
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,2.984226,Analysis Group\n3.8,"Boston, MA",1981,Consulting,Boston,MA,3.635794,-0.651567,MA,Northeast,MA,Northeast,MA,Northeast
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.500000,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing,Newton,MA,3.242113,0.257887,MA,Northeast,MA,Northeast,MA,Northeast
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.900000,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing,New York,NY,3.539615,-0.639615,NY,Northeast,NY,Northeast,NY,Northeast


### **Task 4.2 (Grouped Aggregation)**
Group your newly merged dataframe by the `US_Region` column. For each region, calculate:
1. The **total count** of job postings in that region.
2. The **average (mean)** employee `Rating` in that region.

In [57]:
# Write your solution for Task 4.2 here
df.groupby('US_Region')['Job Description'].transform('count')
df.groupby('US_Region')['Rating'].transform('mean')

0      3.844553
1      3.885203
2      3.844553
3      3.844553
4      3.844553
         ...   
549         NaN
550    3.885203
551    3.844553
552         NaN
553    3.844553
Name: Rating, Length: 554, dtype: float64

### **Task 4.3 (Data Export)**
Save your final, aggregated summary table from Task 4.2 to a new file named `regional_job_summary.csv`.

In [60]:
# Write your solution for Task 4.3 here
df.to_csv('regional_job_summary.csv')